# Reencontra — Testes das regras de SP

Sete casos artificiais, somente em views temporárias, para testar as quatro regras da fonte real.
A validação SQL desta cópia é igual à utilizada no pipeline. Se alterar uma regra, sincronize os dois notebooks.
Esperado: um aprovado, seis rejeitados. Por regra: UF 1, período 1, perfil 1, quantidade 3.
Os casos não são fontes do projeto e não alimentam Bronze, Prata, Ouro ou o JSON do MVP.
Estes testes precisam ser executados no Databricks; esta cópia não contém resultados de execução.


In [0]:
# Casos artificiais apenas para testar; nao gravam nas camadas do projeto
casos = [
    ("Valido", 2024, "SP", 1, "desaparecimento", "adulto", "masculino", "10"),
    ("UF invalida", 2024, "RJ", 1, "desaparecimento", "adulto", "masculino", "10"),
    ("Mes invalido", 2024, "SP", 13, "desaparecimento", "adulto", "masculino", "10"),
    ("Perfil invalido", 2024, "SP", 1, "desaparecimento", "invalido", "masculino", "10"),
    ("Quantidade ausente", 2024, "SP", 1, "desaparecimento", "adulto", "masculino", None),
    ("Quantidade negativa", 2024, "SP", 1, "desaparecimento", "adulto", "masculino", "-1"),
    ("Quantidade fracionaria", 2024, "SP", 1, "desaparecimento", "adulto", "masculino", "1.5"),
]
schema = """caso STRING, ano BIGINT, uf STRING, mes BIGINT,
    indicador STRING, faixa_etaria STRING, sexo STRING,
    quantidade_original STRING"""
spark.createDataFrame(casos, schema).createOrReplaceTempView(
    "teste_entrada_desaparecimentos_sp"
)


In [0]:
spark.sql("""
    CREATE OR REPLACE TEMP VIEW teste_desaparecimentos_sp_validados AS
    WITH preparados AS (
        SELECT *,
            TRY_CAST(quantidade_original AS BIGINT) AS quantidade
        FROM teste_entrada_desaparecimentos_sp
    )
    SELECT *,
        COALESCE(uf = 'SP', FALSE) AS uf_valida,

        COALESCE(
            ano BETWEEN 2023 AND YEAR(CURRENT_DATE())
            AND mes BETWEEN 1 AND 12,
            FALSE
        ) AS periodo_valido,

        COALESCE(
            indicador IN ('desaparecimento', 'localizacao')
            AND faixa_etaria IN (
                'crianca', 'adolescente', 'adulto',
                'idoso', 'nao_informada'
            )
            AND sexo IN (
                'masculino', 'feminino', 'nao_informado'
            ),
            FALSE
        ) AS perfil_valido,

        COALESCE(
            quantidade_original RLIKE '^[0-9]+$'
            AND quantidade IS NOT NULL
            AND quantidade >= 0,
            FALSE
        ) AS quantidade_valida

    FROM preparados
""")


DataFrame[]

In [0]:
resultado = spark.sql("""
    SELECT caso, uf_valida, periodo_valido, perfil_valido, quantidade_valida,
           uf_valida AND periodo_valido AND perfil_valido
               AND quantidade_valida AS aprovado
    FROM teste_desaparecimentos_sp_validados
""")
obtidos = {r["caso"]: r["aprovado"] for r in resultado.collect()}
esperados = {c[0]: c[0] == "Valido" for c in casos}
assert obtidos == esperados, "As regras nao produziram os resultados esperados."
display(resultado.orderBy("caso"))
print("Testes passaram: 1 aprovado e 6 rejeitados.")

display(spark.sql("""
    SELECT 'UF deve ser SP' AS regra, COUNT_IF(NOT uf_valida) AS reprovados
    FROM teste_desaparecimentos_sp_validados
    UNION ALL
    SELECT 'Ano e mes validos', COUNT_IF(NOT periodo_valido)
    FROM teste_desaparecimentos_sp_validados
    UNION ALL
    SELECT 'Indicador e perfil permitidos', COUNT_IF(NOT perfil_valido)
    FROM teste_desaparecimentos_sp_validados
    UNION ALL
    SELECT 'Quantidade inteira e nao negativa', COUNT_IF(NOT quantidade_valida)
    FROM teste_desaparecimentos_sp_validados
"""))


caso,uf_valida,periodo_valido,perfil_valido,quantidade_valida,aprovado
Mes invalido,true,false,true,true,false
Perfil invalido,true,true,false,true,false
Quantidade ausente,true,true,true,false,false
Quantidade fracionaria,true,true,true,false,false
Quantidade negativa,true,true,true,false,false
UF invalida,false,true,true,true,false
Valido,true,true,true,true,true


Testes passaram: 1 aprovado e 6 rejeitados.


regra,reprovados
UF deve ser SP,1
Ano e mes validos,1
Indicador e perfil permitidos,1
Quantidade inteira e nao negativa,3
